# ✊✋✌️ Progetto Finale: Riconoscimento Gesti della Mano
## Sviluppo di una CNN con Data Augmentation per Sasso, Carta, Forbice

**Obiettivo:** costruire il "cervello" visivo di un videogioco interattivo: una Rete Neurale Convoluzionale (CNN) che classifica le immagini catturate dalla webcam in tre categorie: **Sasso (rock)**, **Carta (paper)**, **Forbice (scissors)**.

**Dataset:** *Rock-Paper-Scissors (CGI)*: oltre 2.500 immagini sintetiche 300×300 di mani su sfondo bianco.

| Fase | Contenuto |
|---|---|
| **A** | Data pipeline con `ImageDataGenerator` e Data Augmentation |
| **B** | Architettura CNN sequenziale (150×150×3 → Softmax a 3 classi) |
| **C** | Compilazione, training (15-20 epoche) e callback di stop al 98% |
| **D** | Valutazione sul test set, test reale da webcam e analisi del *domain gap* |

> 💡 Consigliato: eseguire in **Google Colab** con runtime GPU (`Runtime → Cambia tipo di runtime → GPU`).

## 0. Setup e import

In [ ]:
import os
import glob
import random
import zipfile
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# Riproducibilità
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Iperparametri globali
IMG_SIZE = (150, 150)        # dimensione di input richiesta
BATCH_SIZE = 32              # caricamento a batch → RAM sotto controllo
EPOCHS = 20                  # 15-20 epoche come da specifica
CLASSES = ['rock', 'paper', 'scissors']               # ordine fisso delle classi
CLASSES_IT = {'rock': 'Sasso', 'paper': 'Carta', 'scissors': 'Forbice'}

print("TensorFlow:", tf.__version__)
print("GPU disponibili:", tf.config.list_physical_devices('GPU'))

## 1. Download del dataset

Il dataset viene scaricato con `kagglehub`. Poiché la struttura delle cartelle dentro l'archivio Kaggle può cambiare tra le versioni, la funzione `find_class_dirs` cerca automaticamente **tutte le cartelle che contengono le sottocartelle `rock/`, `paper/`, `scissors/`** e assegna a training e test quella giusta in base al nome.

Se Kaggle non è raggiungibile, si usano come fallback i file ufficiali `rps.zip` (training) e `rps-test-set.zip` (test).

In [ ]:
def find_class_dirs(root):
    # Restituisce tutte le cartelle che contengono le sottocartelle rock/paper/scissors
    found = []
    for dirpath, dirnames, _ in os.walk(root):
        if set(CLASSES).issubset({d.lower() for d in dirnames}):
            found.append(dirpath)
    return found


def count_images(folder):
    exts = ('.png', '.jpg', '.jpeg')
    return sum(1 for _, _, files in os.walk(folder) for f in files if f.lower().endswith(exts))


def download_from_kaggle():
    import kagglehub
    path = kagglehub.dataset_download("sanikamal/rock-paper-scissors-dataset")
    print("Path to dataset files:", path)
    return path


def download_fallback(dest='rps_data'):
    # Fallback: archivi ufficiali rps.zip / rps-test-set.zip
    os.makedirs(dest, exist_ok=True)
    urls = {
        'rps.zip': 'https://storage.googleapis.com/learning-datasets/rps.zip',
        'rps-test-set.zip': 'https://storage.googleapis.com/learning-datasets/rps-test-set.zip',
    }
    for name, url in urls.items():
        zpath = os.path.join(dest, name)
        if not os.path.exists(zpath):
            print("Scarico", url)
            urllib.request.urlretrieve(url, zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(dest)
    return dest


try:
    DATA_ROOT = download_from_kaggle()
except Exception as e:
    print("⚠️ Download da Kaggle fallito:", e, "→ uso il fallback")
    DATA_ROOT = download_fallback()

candidates = find_class_dirs(DATA_ROOT)
if len(candidates) < 2:          # struttura inattesa → provo il fallback
    DATA_ROOT = download_fallback()
    candidates = find_class_dirs(DATA_ROOT)

for c in candidates:
    print(f"{count_images(c):5d} immagini  →  {c}")

def pick(cands, keywords, exclude=()):
    for c in cands:
        name = os.path.relpath(c, DATA_ROOT).lower()
        if any(k in name for k in keywords) and not any(x in name for x in exclude):
            return c
    return None

TEST_DIR = pick(candidates, ['test'])
TRAIN_DIR = pick(candidates, ['train', 'rps'], exclude=['test', 'valid'])
# Se i nomi non aiutano: il training set è la cartella più grande
if TRAIN_DIR is None:
    TRAIN_DIR = max([c for c in candidates if c != TEST_DIR], key=count_images)

print("\nTRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

### 1.1 Esplorazione del dataset

In [ ]:
def class_counts(folder):
    return {c: len(os.listdir(os.path.join(folder, c))) for c in CLASSES}

print("Training:", class_counts(TRAIN_DIR))
print("Test    :", class_counts(TEST_DIR))

# Qualche esempio per classe
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for row, cls in enumerate(CLASSES):
    files = sorted(os.listdir(os.path.join(TRAIN_DIR, cls)))
    for col, fname in enumerate(random.sample(files, 5)):
        img = load_img(os.path.join(TRAIN_DIR, cls, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(CLASSES_IT[cls], loc='left', fontsize=13)
plt.suptitle("Esempi dal training set (CGI)", fontsize=15)
plt.tight_layout()
plt.show()

print("Dimensione originale immagini:", img.size)

## Fase A: Data Pipeline e Augmentation

Usiamo `ImageDataGenerator` con `flow_from_directory`, che legge le immagini **a batch direttamente dal disco** (nessun caricamento completo in RAM) e ricava le etichette dalla struttura gerarchica delle sottocartelle (`/rock`, `/paper`, `/scissors`).

**Strategia di split:**
- **Training (80% di `rps`)**: con tutte le trasformazioni di Data Augmentation obbligatorie.
- **Validation (20% di `rps`)**: solo `rescale`, usata durante il training per monitorare l'overfitting e per la callback.
- **Test (`rps-test-set`)**: solo `rescale`, mai visto durante il training, usato **una sola volta** alla fine (Fase D).

> I due generatori di train/validation usano lo stesso `validation_split`: Keras divide i file in modo deterministico (in base all'ordine dei nomi), quindi i due subset sono disgiunti anche se i generatori sono diversi.

In [ ]:
VAL_SPLIT = 0.2

# Generatore di training: normalizzazione + Data Augmentation obbligatoria
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VAL_SPLIT,
)

# Validation e test: SOLO normalizzazione (stesso pre-processing, nessuna augmentation)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)
test_datagen = ImageDataGenerator(rescale=1./255)

common = dict(target_size=IMG_SIZE, batch_size=BATCH_SIZE,
              class_mode='categorical', classes=CLASSES, color_mode='rgb')

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, subset='training', shuffle=True, seed=SEED, **common)

validation_generator = val_datagen.flow_from_directory(
    TRAIN_DIR, subset='validation', shuffle=False, **common)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR, shuffle=False, **common)

print("\nMappatura classi → indici:", train_generator.class_indices)
assert train_generator.class_indices == test_generator.class_indices

### A.1 Visualizzazione della Data Augmentation
La stessa immagine passata più volte nel generatore produce varianti diverse (rotazione, traslazione, shear, flip):

In [ ]:
sample_path = os.path.join(TRAIN_DIR, 'scissors', sorted(os.listdir(os.path.join(TRAIN_DIR, 'scissors')))[0])
x = img_to_array(load_img(sample_path, target_size=IMG_SIZE))[np.newaxis, ...]

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(x[0].astype('uint8')); axes[0].set_title('Originale'); axes[0].axis('off')
aug_iter = train_datagen.flow(x, batch_size=1, seed=SEED)
for i in range(1, 6):
    axes[i].imshow(next(aug_iter)[0])   # già normalizzata in [0,1]
    axes[i].set_title(f'Augmented #{i}'); axes[i].axis('off')
plt.tight_layout(); plt.show()

# Controllo del batch: forma e range dei valori dopo il rescale
xb, yb = next(train_generator)
print("Batch immagini:", xb.shape, "| min/max:", xb.min().round(3), xb.max().round(3))
print("Batch etichette (one-hot):", yb.shape)

## Fase B: Architettura della Rete (CNN)

Rete **sequenziale** con:
1. **Input** 150×150×3 (RGB);
2. **4 blocchi `Conv2D` + `MaxPooling2D`** con filtri crescenti 32 → 64 → 128 → 128: i primi livelli estraggono bordi e texture, i successivi forme sempre più astratte (dita, pugno, palmo);
3. **`Flatten`** + **`Dense(512, relu)`**, preceduto da un `Dropout(0.5)` per ridurre l'overfitting;
4. **Output `Dense(3, softmax)`**: una probabilità per ciascuna classe (somma = 1).

In [ ]:
def build_model():
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

        # Blocco 1
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        # Blocco 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        # Blocco 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        # Blocco 4
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),

        # Classificatore
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dense(len(CLASSES), activation='softmax'),
    ], name='rps_cnn')
    return model

model = build_model()
model.summary()

## Fase C: Compilazione e Training

- **Optimizer:** `RMSprop` (lr = 1e-4 → aggiornamenti stabili con la forte augmentation);
- **Loss:** `categorical_crossentropy` (etichette one-hot, 3 classi);
- **Metrica:** `accuracy`.

**Callback:**
- `StopAt98Accuracy` (personalizzata): interrompe il training appena l'accuratezza supera il **98%**. Per evitare di fermarsi su un modello che "memorizza" il training, la condizione richiede che **sia training sia validation** superino la soglia;
- `EarlyStopping` su `val_loss` (patience 5) con `restore_best_weights`: rete di sicurezza contro l'overfitting;
- `ModelCheckpoint`: salva il miglior modello su disco.

In [ ]:
class StopAt98Accuracy(keras.callbacks.Callback):
    # Ferma il training quando l'accuratezza (train e validation) supera la soglia
    def __init__(self, threshold=0.98):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        acc, val_acc = logs.get('accuracy', 0), logs.get('val_accuracy', 0)
        if acc > self.threshold and val_acc > self.threshold:
            print(f"\n✅ Accuratezza > {self.threshold:.0%} (train={acc:.4f}, val={val_acc:.4f}): stop del training!")
            self.model.stop_training = True


callbacks = [
    StopAt98Accuracy(threshold=0.98),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('rps_best.keras', monitor='val_loss', save_best_only=True, verbose=0),
]

model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1,
)

### C.1 Curve di apprendimento

In [ ]:
h = history.history
epochs_range = range(1, len(h['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs_range, h['accuracy'], 'o-', label='Training')
ax1.plot(epochs_range, h['val_accuracy'], 's-', label='Validation')
ax1.axhline(0.98, color='gray', ls='--', lw=1, label='Soglia 98%')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoca'); ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(epochs_range, h['loss'], 'o-', label='Training')
ax2.plot(epochs_range, h['val_loss'], 's-', label='Validation')
ax2.set_title('Loss'); ax2.set_xlabel('Epoca'); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"Epoche eseguite: {len(epochs_range)}")
print(f"Accuracy finale  → train: {h['accuracy'][-1]:.4f} | val: {h['val_accuracy'][-1]:.4f}")

> **Nota:** è normale che, nelle prime epoche, la *validation accuracy* sia **più alta** della *training accuracy*: le immagini di training sono rese più difficili dall'augmentation (e dal Dropout, attivo solo in training), mentre quelle di validazione sono "pulite".

## Fase D: Test e Validazione "Wild"

### D.1 Validazione sul test set ufficiale
Il test set (`rps-test-set`) non è mai stato visto durante il training e subisce **lo stesso pre-processing** (resize 150×150 + rescale 1/255).

In [ ]:
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n🎯 Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_generator.reset()
probs = model.predict(test_generator, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = test_generator.classes

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=[CLASSES_IT[c] for c in CLASSES]).plot(cmap='Blues', values_format='d')
plt.title('Matrice di confusione: test set CGI')
plt.show()

#### Analisi degli errori
Visualizziamo le immagini del test set classificate in modo errato (se presenti):

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"Errori: {len(wrong)} su {len(y_true)}")

if len(wrong):
    n = min(len(wrong), 10)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.2), squeeze=False)
    for ax, idx in zip(axes[0], wrong[:n]):
        ax.imshow(load_img(test_generator.filepaths[idx], target_size=IMG_SIZE))
        ax.set_title(f"vero: {CLASSES[y_true[idx]]}\npred: {CLASSES[y_pred[idx]]} ({probs[idx].max():.0%})", fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()

### D.2 Funzione di predizione su immagini reali

Per le foto reali applichiamo **esattamente lo stesso pre-processing** del training:
1. **Ritaglio quadrato centrale** (la webcam produce immagini 4:3 o 16:9: un resize diretto a 150×150 deformerebbe la mano);
2. **Resize a 150×150**, conversione in **RGB**;
3. **Normalizzazione 1/255**.

In [ ]:
from PIL import Image, ImageOps

def preprocess_image(path, center_crop=True):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert('RGB')   # orientamento corretto + 3 canali
    if center_crop:
        w, h = img.size
        s = min(w, h)
        img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    img = img.resize(IMG_SIZE)
    arr = np.asarray(img, dtype='float32') / 255.0       # stessa normalizzazione del training
    return img, arr[np.newaxis, ...]


def predict_gesture(path, show=True, center_crop=True):
    img, x = preprocess_image(path, center_crop)
    p = model.predict(x, verbose=0)[0]
    label = CLASSES[int(np.argmax(p))]
    if show:
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.5), gridspec_kw={'width_ratios': [1, 1.3]})
        a1.imshow(img); a1.axis('off')
        a1.set_title(f"Predizione: {CLASSES_IT[label]} ({p.max():.1%})")
        a2.barh([CLASSES_IT[c] for c in CLASSES], p, color=['#888' if c != label else '#2a7' for c in CLASSES])
        a2.set_xlim(0, 1); a2.set_xlabel('Probabilità')
        for i, v in enumerate(p):
            a2.text(min(v + 0.02, 0.9), i, f"{v:.1%}", va='center')
        plt.tight_layout(); plt.show()
    return label, p

# Verifica su un'immagine CGI del test set
label, p = predict_gesture(test_generator.filepaths[0])
print("Etichetta reale:", CLASSES[test_generator.classes[0]])

### D.3 Test reale con la webcam (Google Colab)

Esegui la cella e premi **"Scatta foto"**.

**Consigli per lo scatto** (il modello conosce solo sfondi bianchi puliti):
- usa una **parete chiara o un foglio bianco** come sfondo uniforme;
- illuminazione **diffusa**, senza ombre forti;
- inquadra **solo la mano** (niente viso/braccio), al centro, occupando gran parte dell'inquadratura.

Fuori da Colab, la cella successiva permette di caricare una foto da file.

In [ ]:
from IPython.display import display, Javascript
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.9):
    from google.colab.output import eval_js
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Scatta foto';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    with open(filename, 'wb') as f:
        f.write(b64decode(data.split(',')[1]))
    return filename


try:
    photo = take_photo('my_hand.jpg')
    print("Foto salvata in", photo)
    label, p = predict_gesture(photo)
except ImportError:
    print("Non sei in Colab: usa la cella successiva per caricare una foto da file.")
except Exception as err:
    print("Errore webcam:", err)

In [ ]:
# Alternativa: carica una o più foto (Colab) oppure indica i percorsi locali
REAL_PHOTOS = []   # es. ['foto/sasso.jpg', 'foto/carta.jpg', 'foto/forbice.jpg']

try:
    from google.colab import files
    uploaded = files.upload()
    REAL_PHOTOS += list(uploaded.keys())
except ImportError:
    pass

for path in REAL_PHOTOS:
    predict_gesture(path)

### D.4 Analisi del Domain Gap: CGI vs foto reali

Per un confronto quantitativo, scatta (o carica) alcune foto reali **etichettandole**: ad esempio 3-5 foto per gesto, nominando i file `rock_1.jpg`, `paper_2.jpg`, `scissors_3.jpg`… oppure riempiendo la lista `real_labeled` a mano. La cella confronta **accuratezza** e **confidenza media** sui due domini.

In [ ]:
# Foto reali etichettate: (percorso, classe vera)
real_labeled = []
for path in REAL_PHOTOS + glob.glob('real_photos/*'):
    name = os.path.basename(path).lower()
    for c in CLASSES:
        if name.startswith(c):
            real_labeled.append((path, c))

# Si possono aggiungere scatti webcam a mano, es.:
# real_labeled.append((take_photo('rock_webcam.jpg'), 'rock'))

def summarize(pred_probs, true_idx):
    pred_idx = pred_probs.argmax(axis=1)
    return {
        'accuracy': float((pred_idx == true_idx).mean()),
        'confidenza media': float(pred_probs.max(axis=1).mean()),
        'prob. media classe corretta': float(pred_probs[np.arange(len(true_idx)), true_idx].mean()),
    }

cgi_stats = summarize(probs, y_true)

if real_labeled:
    real_x = np.vstack([preprocess_image(p)[1] for p, _ in real_labeled])
    real_true = np.array([CLASSES.index(c) for _, c in real_labeled])
    real_probs = model.predict(real_x, verbose=0)
    real_stats = summarize(real_probs, real_true)

    print(f"{'Metrica':<30}{'CGI (test)':>12}{'Foto reali':>12}")
    for k in cgi_stats:
        print(f"{k:<30}{cgi_stats[k]:>12.2%}{real_stats[k]:>12.2%}")

    fig, axes = plt.subplots(1, len(real_labeled), figsize=(2.6 * len(real_labeled), 3.2), squeeze=False)
    for ax, (path, c), pr in zip(axes[0], real_labeled, real_probs):
        ax.imshow(preprocess_image(path)[0]); ax.axis('off')
        ok = CLASSES[pr.argmax()] == c
        ax.set_title(f"vero: {c}\npred: {CLASSES[pr.argmax()]} ({pr.max():.0%})", color='green' if ok else 'red', fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print("Nessuna foto reale etichettata: metriche CGI →", {k: f"{v:.2%}" for k, v in cgi_stats.items()})

#### Discussione: perché il modello sbaglia di più sulle foto reali?

Il *domain gap* è la differenza statistica tra le immagini su cui il modello è stato addestrato (**dominio sorgente: CGI**) e quelle su cui viene usato (**dominio target: webcam**). Una CNN impara le regolarità del dataset di training, *incluse quelle che non c'entrano con il gesto*.

| Aspetto | Immagini CGI (training) | Foto reali (webcam) | Effetto sul modello |
|---|---|---|---|
| **Sfondo** | Bianco puro, uniforme | Pareti, ombre, oggetti, texture | Bordi e pattern extra vengono interpretati come parti della mano |
| **Illuminazione** | Perfetta, senza rumore | Ombre, riflessi, bilanciamento del bianco variabile | Contrasti e colori diversi da quelli visti |
| **Pelle / texture** | Rendering liscio, pochi toni | Rughe, peli, unghie, anelli, maniche | Feature di basso livello mai viste |
| **Inquadratura** | Mano centrata, sempre dal basso/lato, braccio parziale | Angolazioni e distanze arbitrarie, possibile viso nel frame | La posizione relativa delle dita cambia |
| **Qualità** | 300×300 nitide | Rumore del sensore, compressione JPEG, mosso | Dettagli fini alterati |

**Osservazioni tipiche:**
- Sul test CGI l'accuratezza è molto alta (spesso > 90-95%), mentre sulle foto reali scende, a volte vicino al caso (33%), con **predizioni molto sicure ma sbagliate**: la softmax non "sa di non sapere".
- Le confusioni più frequenti sono **Carta ↔ Forbice** (dita estese) e la tendenza a predire sempre la stessa classe quando lo sfondo non è bianco.
- Con sfondo chiaro, buona luce e mano centrata, i risultati migliorano sensibilmente: è la conferma che il gap è dovuto soprattutto al **contesto** e non al gesto.

**Come ridurre il domain gap:**
1. **Augmentation più ricca**: `brightness_range`, `zoom_range`, `height_shift_range`, variazioni di colore, sfondi casuali sostituiti al bianco;
2. **Fine-tuning con poche foto reali** (anche 50-100 per classe) aggiunte al training;
3. **Transfer learning** da reti pre-addestrate su foto reali (es. MobileNetV2 / EfficientNet su ImageNet), molto più robuste a texture e sfondi;
4. **Pre-processing in inferenza**: segmentazione/ritaglio della mano (es. MediaPipe Hands) e sostituzione dello sfondo con bianco, così l'input assomiglia al dominio CGI;
5. **Soglia di confidenza** nel videogioco: se la probabilità massima è bassa, chiedere di ripetere il gesto invece di giocare una mossa sbagliata.

## 5. Salvataggio del modello
Il modello viene salvato in formato `.keras`, pronto per essere caricato dal videogioco con `keras.models.load_model`.

In [ ]:
model.save('rps_cnn_final.keras')
print("Modello salvato: rps_cnn_final.keras")

# Esempio di ricaricamento
reloaded = keras.models.load_model('rps_cnn_final.keras')
print("Test accuracy modello ricaricato:", round(reloaded.evaluate(test_generator, verbose=0)[1], 4))

## 6. Conclusioni

- La pipeline con `ImageDataGenerator` carica i dati **a batch** e applica **Data Augmentation** (rotazione 40°, shift orizzontale 20%, shear 20%, flip orizzontale) che rende il modello robusto a posizione e orientamento della mano.
- La CNN a 4 blocchi convoluzionali (32→64→128→128) + Dense(512) + Softmax(3) raggiunge un'accuratezza elevata sul test set CGI in meno di 20 epoche; la callback personalizzata interrompe il training al superamento del 98%.
- Il test "wild" con la webcam evidenzia il **domain gap**: il modello funziona bene solo in condizioni simili al training (sfondo chiaro e uniforme). Per un videogioco reale il passo successivo è il **transfer learning** e/o il **fine-tuning con foto reali**, insieme a una soglia di confidenza sulle predizioni.